# 03 - API Deployment

FastAPI setup, model loading & warmup, batch inference endpoint, streaming response, rate limiting, health checks, monitoring & logging, Docker containerization.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import json
from collections import deque, defaultdict
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. FastAPI Setup (Mock)


In [ ]:
# Simulating FastAPI app structure
class MockFastAPI:
    def __init__(self):
        self.routes = {}
        self.middleware = []
    
    def get(self, path):
        def decorator(func):
            self.routes[f'GET {path}'] = func
            return func
        return decorator
    
    def post(self, path):
        def decorator(func):
            self.routes[f'POST {path}'] = func
            return func
        return decorator
    
    def call_route(self, method, path, **kwargs):
        key = f'{method} {path}'
        if key in self.routes:
            return self.routes[key](**kwargs)
        return {'error': 'Route not found'}

app = MockFastAPI()

@app.get('/health')
def health_check():
    return {'status': 'healthy', 'timestamp': time.time()}

@app.get('/info')
def model_info():
    return {
        'model': 'Llama-2-7b',
        'version': '1.0.0',
        'max_length': 4096,
        'dtype': 'float16'
    }

@app.post('/generate')
def generate(prompt='', max_tokens=100, temperature=0.7):
    # Mock generation
    time.sleep(0.1)
    return {
        'prompt': prompt,
        'generated_text': f'Generated response for: {prompt[:30]}...',
        'tokens_generated': min(max_tokens, len(prompt.split()) + 10),
        'generation_time': 0.1
    }

print('FastAPI routes registered:')
for route in app.routes:
    print(f'  {route}')


## 2. Model Loading & Warmup


In [ ]:
class ModelLoader:
    def __init__(self, model_name='mock-llm'):
        self.model_name = model_name
        self.loaded = False
        self.warmup_complete = False
        self.load_time = 0
    
    def load(self):
        print(f'Loading model: {self.model_name}')
        start = time.time()
        # Simulate loading
        time.sleep(0.5)
        self.loaded = True
        self.load_time = time.time() - start
        print(f'Model loaded in {self.load_time:.2f}s')
        return self
    
    def warmup(self, num_runs=3):
        if not self.loaded:
            raise RuntimeError('Model not loaded')
        print(f'Warming up with {num_runs} inference runs...')
        for i in range(num_runs):
            start = time.time()
            _ = self._mock_inference('warmup prompt', max_tokens=10)
            elapsed = time.time() - start
            print(f'  Warmup run {i+1}: {elapsed:.3f}s')
        self.warmup_complete = True
        print('Warmup complete!')
    
    def _mock_inference(self, prompt, max_tokens=100):
        time.sleep(0.05)
        return 'x' * max_tokens
    
    def status(self):
        return {
            'model': self.model_name,
            'loaded': self.loaded,
            'warmup': self.warmup_complete,
            'load_time': self.load_time
        }

loader = ModelLoader('Llama-2-7b')
loader.load()
loader.warmup(num_runs=3)
print(f'\nStatus: {loader.status()}')


## 3. Batch Inference Endpoint


In [ ]:
class BatchInference:
    def __init__(self, model_loader, max_batch_size=8):
        self.model = model_loader
        self.max_batch_size = max_batch_size
        self.request_queue = deque()
        self.stats = {'total_requests': 0, 'total_batches': 0, 'avg_latency': 0}
    
    def add_request(self, request):
        self.request_queue.append({
            'id': request.get('id', 'unknown'),
            'prompt': request['prompt'],
            'max_tokens': request.get('max_tokens', 100),
            'timestamp': time.time()
        })
    
    def process_batch(self):
        if not self.request_queue:
            return []
        
        batch = []
        while self.request_queue and len(batch) < self.max_batch_size:
            batch.append(self.request_queue.popleft())
        
        start = time.time()
        results = []
        for req in batch:
            result = self._mock_generate(req['prompt'], req['max_tokens'])
            results.append({
                'id': req['id'],
                'result': result,
                'latency': time.time() - req['timestamp']
            })
        
        batch_latency = time.time() - start
        self.stats['total_requests'] += len(batch)
        self.stats['total_batches'] += 1
        self.stats['avg_latency'] = (self.stats['avg_latency'] * (self.stats['total_batches'] - 1) + batch_latency) / self.stats['total_batches']
        
        return results
    
    def _mock_generate(self, prompt, max_tokens):
        time.sleep(0.02 * max_tokens / 10)
        words = prompt.split()
        return ' '.join(words[:3] + ['generated'] * min(5, max_tokens))
    
    def get_stats(self):
        return self.stats

batch_inf = BatchInference(loader, max_batch_size=4)

# Add requests
for i in range(10):
    batch_inf.add_request({
        'id': f'req_{i}',
        'prompt': f'This is prompt number {i}',
        'max_tokens': 20 + i * 5
    })

# Process all
all_results = []
while batch_inf.request_queue:
    results = batch_inf.process_batch()
    all_results.extend(results)
    print(f'Processed batch of {len(results)} requests')

print(f'\nTotal processed: {len(all_results)}')
print(f'Stats: {batch_inf.get_stats()}')


## 4. Streaming Response Simulation


In [ ]:
class StreamingGenerator:
    def __init__(self, chunk_size=5):
        self.chunk_size = chunk_size
    
    def generate_stream(self, prompt, max_tokens=50):
        tokens = ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'and', 'then', 'runs', 'away', 'into', 'the', 'forest', 'happily', 'ever', 'after', 'the', 'end']
        generated = []
        for i in range(0, min(max_tokens, len(tokens)), self.chunk_size):
            chunk = tokens[i:i+self.chunk_size]
            generated.extend(chunk)
            yield {
                'chunk': ' '.join(chunk),
                'generated_so_far': ' '.join(generated),
                'finished': i + self.chunk_size >= min(max_tokens, len(tokens))
            }
            time.sleep(0.05)

streamer = StreamingGenerator(chunk_size=3)
print('Streaming response:')
for chunk in streamer.generate_stream('Hello', max_tokens=15):
    print(f'  Chunk: {chunk["chunk"]} | Finished: {chunk["finished"]}')


## 5. Rate Limiting


In [ ]:
class RateLimiter:
    def __init__(self, max_requests=10, window_seconds=60):
        self.max_requests = max_requests
        self.window = window_seconds
        self.requests = defaultdict(deque)
    
    def is_allowed(self, client_id):
        now = time.time()
        # Remove old requests
        while self.requests[client_id] and self.requests[client_id][0] < now - self.window:
            self.requests[client_id].popleft()
        
        if len(self.requests[client_id]) < self.max_requests:
            self.requests[client_id].append(now)
            return True, len(self.requests[client_id])
        return False, len(self.requests[client_id])
    
    def get_remaining(self, client_id):
        now = time.time()
        while self.requests[client_id] and self.requests[client_id][0] < now - self.window:
            self.requests[client_id].popleft()
        return self.max_requests - len(self.requests[client_id])

limiter = RateLimiter(max_requests=5, window_seconds=10)

# Simulate requests
client = 'user_123'
for i in range(8):
    allowed, count = limiter.is_allowed(client)
    remaining = limiter.get_remaining(client)
    status = 'ALLOWED' if allowed else 'BLOCKED'
    print(f'Request {i+1}: {status} (used: {count}, remaining: {remaining})')


## 6. Health Checks & Monitoring


In [ ]:
class HealthMonitor:
    def __init__(self):
        self.metrics = {
            'requests_total': 0,
            'requests_success': 0,
            'requests_failed': 0,
            'avg_latency': 0,
            'uptime_start': time.time()
        }
        self.latency_history = deque(maxlen=100)
    
    def record_request(self, success=True, latency=0):
        self.metrics['requests_total'] += 1
        if success:
            self.metrics['requests_success'] += 1
        else:
            self.metrics['requests_failed'] += 1
        self.latency_history.append(latency)
        self.metrics['avg_latency'] = np.mean(self.latency_history)
    
    def health_check(self):
        uptime = time.time() - self.metrics['uptime_start']
        error_rate = self.metrics['requests_failed'] / max(self.metrics['requests_total'], 1)
        status = 'healthy' if error_rate < 0.05 else 'degraded' if error_rate < 0.2 else 'unhealthy'
        return {
            'status': status,
            'uptime_seconds': uptime,
            'error_rate': error_rate,
            'avg_latency_ms': self.metrics['avg_latency'] * 1000
            'total_requests': self.metrics['requests_total']
        }
    
    def get_metrics(self):
        return self.metrics

monitor = HealthMonitor()

# Simulate traffic
np.random.seed(42)
for i in range(50):
    latency = np.random.exponential(0.1)
    success = np.random.random() > 0.05
    monitor.record_request(success=success, latency=latency)

print('Health Check:')
print(monitor.health_check())
print(f'\nMetrics: {monitor.get_metrics()}')


## 7. Monitoring Dashboard Visualization


In [ ]:
# Simulate request metrics over time
time_points = np.arange(0, 60, 1)
request_rate = 10 + 5 * np.sin(time_points / 10) + np.random.normal(0, 2, len(time_points))
latency_ms = 50 + 20 * np.sin(time_points / 15 + 1) + np.random.exponential(10, len(time_points))
error_rate = np.maximum(0, 2 + np.sin(time_points / 20) + np.random.normal(0, 1, len(time_points)))

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(time_points, request_rate, color='steelblue', linewidth=2)
axes[0].set_ylabel('Requests/sec')
axes[0].set_title('API Monitoring Dashboard')
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_points, latency_ms, color='orange', linewidth=2)
axes[1].set_ylabel('Latency (ms)')
axes[1].axhline(y=100, color='red', linestyle='--', alpha=0.5, label='SLA Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(time_points, error_rate, color='red', linewidth=2)
axes[2].set_ylabel('Error Rate (%)')
axes[2].set_xlabel('Time (seconds)')
axes[2].axhline(y=5, color='gray', linestyle='--', alpha=0.5, label='Alert Threshold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Docker Containerization


In [ ]:
dockerfile = '''
# Dockerfile for LLM API Deployment
FROM python:3.10-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \
  CMD curl -f http://localhost:8000/health || exit 1

# Run application
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements = '''
fastapi==0.104.1
uvicorn[standard]==0.24.0
numpy
torch>=2.0.0
transformers>=4.35.0
pydantic
prometheus-client
'''

print('=== Dockerfile ===')
print(dockerfile)
print('\n=== requirements.txt ===')
print(requirements)


## 9. Exercises


In [ ]:
# Exercise 1: Implement request queue with priority
# Exercise 2: Add authentication middleware simulation
# Exercise 3: Build a load testing script
# Exercise 4: Implement model A/B testing in API
print('Exercises listed!')
